# 🖼️ Datathon Template — IMAGE DATA (Computer Vision, Image Classification)

| # | Step | কি করব |
|---|------|--------|
| 0 | Setup & Config | path, model name, image size, epochs |
| 1 | Load metadata | CSV format (`image,label`) অথবা folder format (`train/<class>/img.jpg`) |
| 2 | EDA | class distribution, sample grid, image size, corrupted/duplicate, channel mean/std, brightness |
| 3 | Preprocessing & Augmentation | resize, normalize, flip, rotate, color jitter, random erasing |
| 4 | ⚡ Quick Baseline | classical features + pretrained CNN embeddings → Logistic Regression |
| 5 | Dataset / DataLoader | PyTorch |
| 6 | Model | `timm` pretrained (ResNet / EfficientNet / ConvNeXt / ViT) — transfer learning |
| 7 | Training | AdamW + OneCycle, AMP, label smoothing, Mixup, early stopping, K-Fold |
| 8 | Evaluation | learning curve, confusion matrix, report, misclassified images |
| 9 | Interpretation | Grad-CAM — model ছবির কোথায় দেখছে |
| 10 | Inference + TTA + Submission | fold ensemble + flip TTA |
| 11 | Other CV tasks | multi-label, regression, detection, segmentation |

> **GPU চালু করো** (Kaggle: Settings → Accelerator → GPU T4/P100; Colab: Runtime → Change runtime type → GPU)।
> Pretrained weight download এর জন্য Kaggle এ **Internet ON** রাখতে হবে।
> কোনো data না দিলে notebook নিজেই একটা ছোট **shapes dataset** (circle / square / triangle) বানিয়ে নেবে।

## ☁️ Google Colab Setup (সবার আগে এটা চালাও)

**GPU চালু করা:** `Runtime → Change runtime type → T4 GPU → Save` (image / transformer এর জন্য জরুরি)

**Data আনার ৪টা উপায় — নিচের `DATA_SOURCE` বদলাও:**
| `DATA_SOURCE` | কিভাবে |
|---|---|
| `"demo"` | কিছু লাগবে না — notebook নিজেই practice data বানাবে |
| `"upload"` | Cell চালালে **Choose Files** button আসবে → `train.csv`, `test.csv`, `sample_submission.csv` (বা একটা `.zip`) select করো |
| `"drive"` | আগে Google Drive এ data রাখো (যেমন `MyDrive/datathon/image/`) → `DRIVE_DATA_DIR` ঠিক করো → permission দাও |
| `"kaggle"` | Kaggle → Settings → **Create New Token** → `kaggle.json` download → cell চালালে upload করো; competition page এ আগে **rules accept** করতে হবে |

সব data শেষে `data/image` folder এ আসবে, `.zip` থাকলে auto unzip হবে — তাই নিচের CONFIG এ path বদলাতে হবে না
(শুধু file / column এর নাম মিলিয়ে নিও)।

> ⚠️ Colab বন্ধ হলে `/content` এর সব file মুছে যায় — submission আর model notebook এর শেষ cell দিয়ে download / Drive এ save করো।

In [ ]:
import sys, os, shutil, zipfile, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ---------- 1) যে library Colab এ নেই সেগুলো install ----------
NEED = {"timm": "timm", "seaborn": "seaborn"}          # {import_name: pip_name}
missing = [pip for mod, pip in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

# ---------- 2) Data ----------
DATA_SOURCE = "demo"                                  # "demo" | "upload" | "drive" | "kaggle"
DRIVE_DATA_DIR = "/content/drive/MyDrive/datathon/image"
KAGGLE_COMPETITION = ""                               # e.g. "titanic"  (URL: kaggle.com/competitions/<এই নাম>)
KAGGLE_DATASET = ""                                   # competition না হয়ে dataset হলে: "owner/dataset-name"
LOCAL_DATA_DIR = Path("data/image")

def unzip_all(folder):
    while zips := list(Path(folder).rglob("*.zip")):                       # zip এর ভেতরে zip থাকলেও (Kaggle এ common)
        for z in zips:
            print("Unzipping", z.name)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent)
            z.unlink()

if IN_COLAB and DATA_SOURCE != "demo":
    from google.colab import files, drive
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if DATA_SOURCE == "upload":
        for name in files.upload():
            shutil.move(name, LOCAL_DATA_DIR / name)
    elif DATA_SOURCE == "drive":
        drive.mount("/content/drive")
        shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)   # local copy → Drive থেকে পড়ার চেয়ে অনেক দ্রুত
    elif DATA_SOURCE == "kaggle":
        # নতুন token (KGAT_...) বা পুরনো kaggle.json — যেটা আছে সেটা দিয়েই চলবে
        import getpass
        if not os.environ.get("KAGGLE_API_TOKEN") and not Path("~/.kaggle/kaggle.json").expanduser().exists():
            tok = getpass.getpass("Kaggle API token paste করো (লুকানো থাকবে, KGAT_... দিয়ে শুরু) অথবা খালি রেখে Enter দিয়ে kaggle.json upload করো: ").strip()
            if tok:
                os.environ["KAGGLE_API_TOKEN"] = tok
            else:
                files.upload()
                Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
                shutil.move("kaggle.json", Path("~/.kaggle/kaggle.json").expanduser())
                os.chmod(Path("~/.kaggle/kaggle.json").expanduser(), 0o600)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=False)
        if KAGGLE_COMPETITION:
            subprocess.run(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(LOCAL_DATA_DIR)], check=True)
        else:
            subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(LOCAL_DATA_DIR)], check=True)
    unzip_all(LOCAL_DATA_DIR)

if LOCAL_DATA_DIR.exists():
    print(f"\n📁 {LOCAL_DATA_DIR} এর ভেতরে:")
    for p in sorted(LOCAL_DATA_DIR.iterdir())[:30]:
        print("  ", p.name, "/" if p.is_dir() else f"({p.stat().st_size / 1e6:.2f} MB)")
print("IN_COLAB =", IN_COLAB, "| DATA_SOURCE =", DATA_SOURCE)

## 0. Setup & Config

In [ ]:
import os, random, time, hashlib, importlib, warnings, copy
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import v2 as T

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn import metrics

def try_import(name):
    try:
        return importlib.import_module(name)
    except ImportError:
        print(f"⚠️  '{name}' নেই → pip install {name}")
        return None

timm = try_import("timm")
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

In [ ]:
# ======================= CONFIG =======================
DATA_DIR = Path("data/image")                  # Kaggle: Path("/kaggle/input/<competition>")
DATA_FORMAT = "csv"                            # "csv" → train.csv (image,label) | "folder" → train/<class_name>/*.jpg
TRAIN_CSV = DATA_DIR / "train.csv"
TRAIN_IMG_DIR = DATA_DIR / "train"
TEST_IMG_DIR = DATA_DIR / "test"
SUBMISSION_FILE = DATA_DIR / "sample_submission.csv"
OUTPUT_DIR = Path("outputs/image")

IMG_COL, LABEL_COL = "image", "label"          # CSV এর column নাম
IMG_EXT = ""                                   # CSV তে extension না থাকলে ".jpg" দাও

MODEL_NAME = "resnet18"     # timm: "efficientnet_b0", "convnext_tiny", "resnet50", "tf_efficientnetv2_s", "vit_small_patch16_224"
PRETRAINED = True
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-2
LABEL_SMOOTHING = 0.1
USE_MIXUP = False            # ছোট dataset এ overfit কমায়
PATIENCE = 3                 # early stopping
N_FOLDS = 5
TRAIN_FOLDS = [0]            # সময় থাকলে [0,1,2,3,4] → fold ensemble
TTA = True
SEED = 42
NUM_WORKERS = 0 if os.name == "nt" else 2      # Windows notebook এ 0 রাখো
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
USE_AMP = DEVICE.type == "cuda"
# ======================================================

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| torch", torch.__version__, "| timm", getattr(timm, "__version__", None))
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
elif IN_COLAB:
    print("⚠️ GPU চালু নেই! Runtime → Change runtime type → T4 GPU → তারপর আবার Run all")

### (Practice) Demo dataset generator

In [ ]:
def make_demo_images(root, n_train=600, n_test=150, size=96, seed=SEED):
    rng = np.random.default_rng(seed)
    classes = ["circle", "square", "triangle"]

    def draw(cls):
        img = Image.new("RGB", (size, size), tuple(int(v) for v in rng.integers(0, 120, 3)))
        d = ImageDraw.Draw(img)
        col = tuple(int(v) for v in rng.integers(130, 256, 3))
        r = int(rng.integers(size // 6, size // 3)); cx, cy = (int(v) for v in rng.integers(r, size - r, 2))
        if cls == "circle":
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=col)
        elif cls == "square":
            d.rectangle([cx - r, cy - r, cx + r, cy + r], fill=col)
        else:
            d.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=col)
        arr = np.array(img).astype(np.int16) + rng.normal(0, 15, (size, size, 3)).astype(np.int16)
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    (root / "train").mkdir(parents=True, exist_ok=True); (root / "test").mkdir(parents=True, exist_ok=True)
    rows = []
    for i in range(n_train):
        cls = str(rng.choice(classes, p=[.45, .35, .20]))          # একটু imbalance
        draw(cls).save(root / "train" / f"train_{i:04d}.png"); rows.append((f"train_{i:04d}.png", cls))
    pd.DataFrame(rows, columns=["image", "label"]).to_csv(root / "train.csv", index=False)
    names = []
    for i in range(n_test):
        draw(str(rng.choice(classes))).save(root / "test" / f"test_{i:04d}.png"); names.append(f"test_{i:04d}.png")
    pd.DataFrame({"image": names, "label": classes[0]}).to_csv(root / "sample_submission.csv", index=False)

DEMO = not (TRAIN_CSV.exists() or (DATA_FORMAT == "folder" and TRAIN_IMG_DIR.exists()))
if DEMO:
    print("Data পাওয়া যায়নি → DEMO shapes dataset বানানো হচ্ছে")
    make_demo_images(DATA_DIR)
    DATA_FORMAT, IMG_SIZE, EPOCHS, BATCH_SIZE = "csv", 96, 4, 32      # CPU তেও দ্রুত চলবে

## 1. Load metadata

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

if DATA_FORMAT == "csv":
    df = pd.read_csv(TRAIN_CSV)
    df["path"] = df[IMG_COL].astype(str).apply(lambda x: str(TRAIN_IMG_DIR / f"{x}{IMG_EXT}"))
else:
    rows = [(str(p), p.parent.name) for p in sorted(TRAIN_IMG_DIR.rglob("*")) if p.suffix.lower() in IMG_EXTS]
    df = pd.DataFrame(rows, columns=["path", LABEL_COL])
    df[IMG_COL] = df["path"].apply(lambda p: Path(p).name)

sample_sub = pd.read_csv(SUBMISSION_FILE) if SUBMISSION_FILE.exists() else None
if sample_sub is not None and IMG_COL in sample_sub:
    test_df = sample_sub[[IMG_COL]].copy()                      # submission এর order ঠিক রাখতে
else:
    test_df = pd.DataFrame({IMG_COL: sorted(p.name for p in TEST_IMG_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS)})
test_df["path"] = test_df[IMG_COL].astype(str).apply(lambda x: str(TEST_IMG_DIR / f"{x}{IMG_EXT if not Path(x).suffix else ''}"))

missing_files = (~df["path"].apply(os.path.exists)).sum()
print(f"Train: {len(df)} images | Test: {len(test_df)} | Missing files: {missing_files}")
df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)

LE = LabelEncoder().fit(df[LABEL_COL])
df["y"] = LE.transform(df[LABEL_COL])
CLASSES = list(LE.classes_)
N_CLASSES = len(CLASSES)
print("Classes:", CLASSES)
display(df.head())

## 2. EDA
### 2.1 Class distribution

In [ ]:
vc = df[LABEL_COL].value_counts()
fig, ax = plt.subplots(figsize=(max(6, .5 * N_CLASSES), 4))
sns.barplot(x=vc.index.astype(str), y=vc.values, ax=ax)
for i, v in enumerate(vc.values):
    ax.text(i, v, f"{v}\n{v / len(df):.1%}", ha="center", va="bottom", fontsize=8)
ax.set_title(f"Class distribution (imbalance ratio = {vc.max() / vc.min():.2f})"); plt.xticks(rotation=45); plt.show()

### 2.2 Sample images per class

In [ ]:
def show_grid(paths, titles=None, ncols=6, size=2.2):
    nrows = int(np.ceil(len(paths) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(size * ncols, size * nrows))
    for ax in np.array(axes).reshape(-1):
        ax.axis("off")
    for ax, p, t in zip(np.array(axes).reshape(-1), paths, titles or [""] * len(paths)):
        ax.imshow(Image.open(p).convert("RGB")); ax.set_title(str(t), fontsize=8)
    plt.tight_layout(); plt.show()

per_class = 6
samp = pd.concat([g.sample(min(per_class, len(g)), random_state=SEED) for _, g in df.groupby(LABEL_COL)])
show_grid(samp["path"].tolist(), samp[LABEL_COL].tolist(), ncols=per_class)

### 2.3 Image size / mode / file size / corrupted / duplicate

In [ ]:
def image_meta(path):
    try:
        with Image.open(path) as im:
            w, h = im.size; mode = im.mode
            im.verify()
        ok = True
    except Exception:
        w = h = mode = None; ok = False
    return {"width": w, "height": h, "mode": mode, "ok": ok, "kb": os.path.getsize(path) / 1024,
            "md5": hashlib.md5(open(path, "rb").read()).hexdigest()}

meta_sample = df.sample(min(3000, len(df)), random_state=SEED)
meta = pd.DataFrame([image_meta(p) for p in meta_sample["path"]], index=meta_sample.index)
df_meta = meta_sample.join(meta)

print("Corrupted images:", (~df_meta["ok"]).sum())
print("Modes:", df_meta["mode"].value_counts().to_dict())
dup = df_meta[df_meta.duplicated("md5", keep=False)]
print(f"Duplicate images (same bytes): {len(dup)}  | same image different label: "
      f"{(dup.groupby('md5')[LABEL_COL].nunique() > 1).sum()}")
display(df_meta[["width", "height", "kb"]].describe().T)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].scatter(df_meta["width"], df_meta["height"], s=6, alpha=.4); axes[0].set(xlabel="width", ylabel="height", title="Image sizes")
sns.histplot(df_meta["width"] / df_meta["height"], bins=40, ax=axes[1]); axes[1].set_title("Aspect ratio (w/h)")
sns.histplot(df_meta["kb"], bins=40, ax=axes[2]); axes[2].set_title("File size (KB)")
plt.tight_layout(); plt.show()

### 2.4 Channel mean/std + brightness per class
Pretrained model হলে **ImageNet mean/std** ব্যবহার করো। Scratch থেকে train করলে নিজের data এর mean/std।

In [ ]:
def img_stats(path, s=64):
    a = np.asarray(Image.open(path).convert("RGB").resize((s, s)), dtype=np.float32) / 255.
    return a.mean((0, 1)), a.std((0, 1))

stats = [img_stats(p) for p in df["path"].sample(min(800, len(df)), random_state=SEED)]
DATA_MEAN = np.mean([m for m, _ in stats], 0); DATA_STD = np.mean([s for _, s in stats], 0)
print("Dataset mean:", DATA_MEAN.round(3), "std:", DATA_STD.round(3))

bright = df.sample(min(1500, len(df)), random_state=SEED).copy()
bright["brightness"] = [img_stats(p)[0].mean() for p in bright["path"]]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=LABEL_COL, y="brightness", data=bright, ax=axes[0]); axes[0].set_title("Brightness per class")
# প্রতিটা class এর "average image" — pattern বোঝা যায়
avg_imgs = [np.mean([np.asarray(Image.open(p).convert("RGB").resize((64, 64)), np.float32)
                     for p in df[df.y == c]["path"].head(100)], 0) / 255 for c in range(min(N_CLASSES, 8))]
axes[1].imshow(np.hstack(avg_imgs)); axes[1].set_title("Mean image per class: " + ", ".join(map(str, CLASSES[:8]))); axes[1].axis("off")
plt.tight_layout(); plt.show()

## 3. Preprocessing & Augmentation
- **Train:** RandomResizedCrop, Flip, Rotation, ColorJitter, RandomErasing → model কে robust করে
- **Valid/Test:** শুধু Resize + Normalize (কোনো random জিনিস না)
- ⚠️ Domain বুঝে augmentation: digit/text এ horizontal flip **ক্ষতিকর**, medical X-ray তে color jitter কম
- আরো শক্তিশালী augmentation লাগলে: `albumentations` library

In [ ]:
MEAN, STD = ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) if PRETRAINED else (tuple(DATA_MEAN), tuple(DATA_STD))

train_tfms = T.Compose([
    T.ToImage(),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), antialias=True),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=.2, contrast=.2, saturation=.2, hue=.03),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(MEAN, STD),
    T.RandomErasing(p=0.2),
])
valid_tfms = T.Compose([
    T.ToImage(),
    T.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(MEAN, STD),
])

def denorm(x):
    return (x.permute(1, 2, 0).numpy() * np.array(STD) + np.array(MEAN)).clip(0, 1)

# Augmentation কেমন দেখায়
img0 = Image.open(df["path"].iloc[0]).convert("RGB")
fig, axes = plt.subplots(1, 8, figsize=(18, 2.6))
axes[0].imshow(img0); axes[0].set_title("original")
for ax in axes[1:]:
    ax.imshow(denorm(train_tfms(img0))); ax.set_title("augmented")
for ax in axes: ax.axis("off")
plt.show()

## 4. ⚡ Quick Baselines (deep training এর আগে)
### 4.1 Classical: resized pixels + color histogram → LogisticRegression / RandomForest

In [ ]:
def classical_features(path, s=32):
    im = Image.open(path).convert("RGB")
    small = np.asarray(im.convert("L").resize((s, s)), np.float32).ravel() / 255.
    hist = np.concatenate([np.histogram(np.asarray(im)[..., c], bins=16, range=(0, 255), density=True)[0] for c in range(3)])
    return np.concatenate([small, hist])

Xc = np.stack([classical_features(p) for p in df["path"]])
skf = StratifiedKFold(5, shuffle=True, random_state=SEED)
for name, clf in {"LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                  "RandomForest": RandomForestClassifier(300, n_jobs=-1, random_state=SEED)}.items():
    print(f"Classical {name:12s} acc = {cross_val_score(clf, Xc, df.y, cv=skf, scoring='accuracy').mean():.4f}")

### 4.2 Pretrained CNN embeddings + LogisticRegression
Backbone freeze করে feature বের করি → উপরে simple classifier। **ছোট data / কম GPU সময়ে খুব শক্তিশালী।**

In [ ]:
class ImageDS(Dataset):
    def __init__(self, paths, labels=None, tfms=None):
        self.paths, self.labels, self.tfms = list(paths), (None if labels is None else list(labels)), tfms
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        x = self.tfms(Image.open(self.paths[i]).convert("RGB"))
        return x if self.labels is None else (x, torch.tensor(self.labels[i], dtype=torch.long))

def make_loader(paths, labels=None, tfms=valid_tfms, shuffle=False, bs=BATCH_SIZE):
    return DataLoader(ImageDS(paths, labels, tfms), batch_size=bs, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda", drop_last=shuffle and len(paths) > bs)

def build_model(num_classes=N_CLASSES, name=MODEL_NAME, pretrained=PRETRAINED):
    """timm থাকলে timm, না হলে torchvision resnet18। Weight download fail করলে random init।"""
    if timm is not None:
        try:
            return timm.create_model(name, pretrained=pretrained, num_classes=num_classes)
        except Exception as e:
            print(f"⚠️ pretrained load failed ({type(e).__name__}) → random init"); return timm.create_model(name, pretrained=False, num_classes=num_classes)
    try:
        m = torchvision.models.resnet18(weights="DEFAULT" if pretrained else None)
    except Exception as e:
        print(f"⚠️ pretrained load failed ({type(e).__name__}) → random init"); m = torchvision.models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes) if num_classes else nn.Identity()
    return m

@torch.no_grad()
def extract_embeddings(model, paths):
    model.eval().to(DEVICE)
    return np.concatenate([model(x.to(DEVICE)).float().cpu().numpy() for x in make_loader(paths)])

backbone = build_model(num_classes=0)                         # num_classes=0 → classifier head বাদ, শুধু feature
emb = extract_embeddings(backbone, df["path"])
print("Embedding shape:", emb.shape)
emb_clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5))
print(f"CNN-embedding + LogReg acc = {cross_val_score(emb_clf, emb, df.y, cv=skf, scoring='accuracy').mean():.4f}")
del backbone

## 5–7. Fine-tuning (Transfer Learning) — K-Fold training loop
- **Optimizer:** AdamW, **Scheduler:** OneCycleLR (warmup + cosine decay)
- **AMP** (mixed precision) → GPU তে ~2x দ্রুত
- **Label smoothing** + **Mixup** → overfitting কমায়
- **Early stopping** + best checkpoint save
- Class imbalance বেশি হলে `CrossEntropyLoss(weight=class_weights)`

In [ ]:
def mixup(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def train_one_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train(); tot, n = 0., 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            if USE_MIXUP and random.random() < 0.5:
                xm, ya, yb, lam = mixup(x, y)
                out = model(xm); loss = lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)
            else:
                loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tot += loss.item() * len(x); n += len(x)
    return tot / n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); tot, probs, ys = 0., [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            out = model(x)
        tot += criterion(out.float(), y).item() * len(x)
        probs.append(out.float().softmax(1).cpu()); ys.append(y.cpu())
    probs, ys = torch.cat(probs).numpy(), torch.cat(ys).numpy()
    return tot / len(ys), probs, ys

def fit_fold(fold, tr_df, va_df):
    seed_everything(SEED + fold)
    model = build_model().to(DEVICE)
    tl = make_loader(tr_df["path"], tr_df["y"], train_tfms, shuffle=True)
    vl = make_loader(va_df["path"], va_df["y"], valid_tfms)
    # class_weights = torch.tensor(len(tr_df) / (N_CLASSES * np.bincount(tr_df.y, minlength=N_CLASSES)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)  # , weight=class_weights
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, total_steps=EPOCHS * len(tl), pct_start=0.15)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    hist, best, best_probs, bad = [], -1, None, 0
    ckpt = OUTPUT_DIR / f"{MODEL_NAME}_fold{fold}.pt"
    for ep in range(EPOCHS):
        t0 = time.time()
        tr_loss = train_one_epoch(model, tl, optimizer, scheduler, criterion, scaler)
        va_loss, probs, ys = evaluate(model, vl, criterion)
        acc = metrics.accuracy_score(ys, probs.argmax(1)); f1 = metrics.f1_score(ys, probs.argmax(1), average="macro")
        hist.append({"epoch": ep + 1, "train_loss": tr_loss, "val_loss": va_loss, "val_acc": acc, "val_f1": f1})
        flag = ""
        if f1 > best:
            best, best_probs, bad, flag = f1, probs, 0, " ✅ saved"
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
        print(f"fold {fold} | ep {ep + 1:02d} | train {tr_loss:.4f} | val {va_loss:.4f} | acc {acc:.4f} | f1 {f1:.4f} | {time.time() - t0:.0f}s{flag}")
        if bad >= PATIENCE:
            print("Early stopping"); break
    return pd.DataFrame(hist), best_probs, ckpt

In [ ]:
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
df["fold"] = -1
for f, (_, va) in enumerate(skf.split(df, df.y)):
    df.loc[va, "fold"] = f

HISTORY, CKPTS = {}, []
oof = np.full((len(df), N_CLASSES), np.nan)
for fold in TRAIN_FOLDS:
    tr_df, va_df = df[df.fold != fold], df[df.fold == fold]
    HISTORY[fold], probs, ckpt = fit_fold(fold, tr_df, va_df)
    oof[va_df.index] = probs; CKPTS.append(ckpt)

## 8. Evaluation
### 8.1 Learning curves (overfit/underfit বোঝা)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for fold, h in HISTORY.items():
    axes[0].plot(h.epoch, h.train_loss, "o-", label=f"train f{fold}"); axes[0].plot(h.epoch, h.val_loss, "s--", label=f"val f{fold}")
    axes[1].plot(h.epoch, h.val_acc, "o-", label=f"acc f{fold}"); axes[1].plot(h.epoch, h.val_f1, "s--", label=f"f1 f{fold}")
axes[0].set_title("Loss"); axes[1].set_title("Validation metrics"); axes[0].legend(); axes[1].legend(); plt.show()

### 8.2 Confusion matrix + classification report (OOF)

In [ ]:
mask = ~np.isnan(oof).any(1)
y_true, y_prob = df.y.values[mask], oof[mask]
y_pred = y_prob.argmax(1)
print(f"OOF accuracy = {metrics.accuracy_score(y_true, y_pred):.4f} | macro-F1 = {metrics.f1_score(y_true, y_pred, average='macro'):.4f}")
print(metrics.classification_report(y_true, y_pred, labels=range(N_CLASSES), target_names=[str(c) for c in CLASSES], digits=4, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(max(12, N_CLASSES), max(5, .45 * N_CLASSES)))
metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=range(N_CLASSES), display_labels=CLASSES, ax=axes[0], cmap="Blues", colorbar=False, xticks_rotation=45)
metrics.ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=range(N_CLASSES), display_labels=CLASSES, ax=axes[1], cmap="Greens", normalize="true", colorbar=False, xticks_rotation=45, values_format=".2f")
plt.tight_layout(); plt.show()

### 8.3 ভুল prediction গুলো দেখো (সবচেয়ে confident ভুল আগে)
Error analysis থেকে বোঝা যায়: label noise? কঠিন class? augmentation দরকার?

In [ ]:
err = df[mask].assign(pred=y_pred, conf=y_prob.max(1))
err = err[err.y != err.pred].sort_values("conf", ascending=False).head(18)
if len(err):
    show_grid(err["path"].tolist(), [f"true={CLASSES[t]}\npred={CLASSES[p]} ({c:.2f})" for t, p, c in zip(err.y, err.pred, err.conf)])
else:
    print("কোনো ভুল নেই 🎉")

## 9. Grad-CAM — model ছবির কোন অংশ দেখে সিদ্ধান্ত নিচ্ছে?
(CNN এর জন্য; ViT এ শেষ conv layer নেই)

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model.eval()
        convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
        assert convs, "Conv layer নেই (ViT?) — Grad-CAM skip"
        convs[-1].register_forward_hook(self._fwd)
    def _fwd(self, m, i, o):
        self.acts = o
        o.register_hook(lambda g: setattr(self, "grads", g))
    def __call__(self, x, cls=None):
        x = x.unsqueeze(0).to(DEVICE).requires_grad_(True)
        out = self.model(x)
        cls = int(out.argmax(1)) if cls is None else cls
        self.model.zero_grad(); out[0, cls].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear").squeeze().detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), cls

cam_model = build_model(pretrained=False).to(DEVICE)
cam_model.load_state_dict(torch.load(CKPTS[0], map_location=DEVICE))
try:
    gc = GradCAM(cam_model)
    rows = df[df.fold == TRAIN_FOLDS[0]].sample(min(6, len(df)), random_state=SEED)
    fig, axes = plt.subplots(2, len(rows), figsize=(2.6 * len(rows), 5.4))
    for j, (_, r) in enumerate(rows.iterrows()):
        x = valid_tfms(Image.open(r.path).convert("RGB"))
        cam, cls = gc(x)
        axes[0, j].imshow(denorm(x)); axes[0, j].set_title(f"true: {r[LABEL_COL]}", fontsize=8)
        axes[1, j].imshow(denorm(x)); axes[1, j].imshow(cam, cmap="jet", alpha=.45); axes[1, j].set_title(f"pred: {CLASSES[cls]}", fontsize=8)
        axes[0, j].axis("off"); axes[1, j].axis("off")
    plt.tight_layout(); plt.show()
except AssertionError as e:
    print(e)

## 10. Inference (fold ensemble + TTA) → Submission

In [ ]:
@torch.no_grad()
def predict(model, paths, tta=TTA):
    model.eval(); out = []
    for x in make_loader(paths):
        x = x.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            p = model(x).float().softmax(1)
            if tta:
                p = (p + model(torch.flip(x, dims=[3])).float().softmax(1)) / 2     # horizontal flip TTA
        out.append(p.cpu())
    return torch.cat(out).numpy()

test_probs = np.zeros((len(test_df), N_CLASSES))
for ckpt in CKPTS:
    m = build_model(pretrained=False).to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    test_probs += predict(m, test_df["path"]) / len(CKPTS)

test_df["pred"] = LE.inverse_transform(test_probs.argmax(1))
sub = pd.DataFrame({IMG_COL: test_df[IMG_COL], LABEL_COL: test_df["pred"]})
if sample_sub is not None:
    sub.columns = sample_sub.columns[:2]
    assert len(sub) == len(sample_sub)
sub.to_csv(OUTPUT_DIR / "submission.csv", index=False)
np.save(OUTPUT_DIR / "test_probs.npy", test_probs)          # পরে অন্য model এর সাথে ensemble করার জন্য
print("Saved →", OUTPUT_DIR / "submission.csv")
display(sub.head()); print(sub.iloc[:, 1].value_counts(normalize=True))

## ☁️ Colab: Submission download + Drive এ backup

In [ ]:
SAVE_TO_DRIVE = False                                  # True → outputs folder Google Drive এ copy হবে
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/datathon/outputs/image"

if IN_COLAB:
    from google.colab import files, drive
    sub_path = OUTPUT_DIR / "submission.csv"
    if sub_path.exists():
        files.download(str(sub_path))                  # browser এ download হবে
    if SAVE_TO_DRIVE:
        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print("Saved to Drive →", DRIVE_OUTPUT_DIR)
else:
    print("Local run — files আছে:", OUTPUT_DIR.resolve())

## 11. Other Computer Vision task হলে কি বদলাবো?

| Task | পরিবর্তন |
|---|---|
| **Multi-label** (একটা ছবিতে একাধিক label) | `num_classes = n_labels`, loss = `nn.BCEWithLogitsLoss()`, prob = `sigmoid`, threshold tune (0.5 না), metric = F1-micro/mAP |
| **Regression** (age, count, score) | `num_classes=1`, loss = `nn.MSELoss()` / `nn.L1Loss()` / `nn.SmoothL1Loss()`, output `.squeeze(1)` |
| **Object detection** | `pip install ultralytics` → `YOLO("yolov8n.pt").train(data="data.yaml", epochs=50, imgsz=640)` |
| **Segmentation** | `pip install segmentation-models-pytorch` → `smp.Unet("resnet34", encoder_weights="imagenet", classes=N)`, loss = Dice + BCE |
| **Grayscale image** (X-ray, MNIST) | `Image.convert("RGB")` করলেই pretrained model চলবে; অথবা `timm.create_model(..., in_chans=1)` |
| **খুব ছোট dataset** | Section 4.2 (embeddings + LogReg) প্রায়ই fine-tune এর সমান ভালো; strong augmentation |
| **অনেক বড় image** (satellite, pathology) | tile/patch এ ভাগ করে train, prediction aggregate |

## ✅ Score বাড়ানোর checklist
1. ভালো backbone: `efficientnet_b0 → b3`, `convnext_tiny/small`, `eva02`, `tf_efficientnetv2_s`
2. **Image size বাড়াও** (224 → 384/512) — প্রায়ই সবচেয়ে বড় gain
3. সব fold train করে **fold ensemble** + TTA
4. ভিন্ন architecture এর `test_probs.npy` average করো (CNN + ViT)
5. LR finder / lower LR for backbone, higher for head (discriminative LR)
6. Error analysis দেখে augmentation বদলাও; label noise থাকলে label smoothing বাড়াও